# UC-TILES-1 — Vector Tiles from OGC Features

**Persona:** Web Map Developer

**Goal:** Ingest a set of administrative polygon features via OGC Features POST,
retrieve the Tile Matrix Sets listing, fetch one MVT tile for a known z/x/y,
verify the content-type is `application/vnd.mapbox-vector-tile`, and finish
with a deep link to the map viewer.

**Prerequisites:**
- GeoID platform running at `DYNASTORE_BASE_URL` (default `http://localhost:8080`)
- `tiles` and `features` extensions enabled in the active SCOPE
- Write-scoped JWT in `DYNASTORE_TOKEN` (or IDP client credentials set)

**References:**
- OGC API — Tiles 1.0 (OGC 20-057)
- Routes: `/tiles/tileMatrixSets`, `/tiles/catalogs/{dataset}/tiles/{z}/{x}/{y}.mvt`

In [ ]:
import json
import os

import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")

# Auto-provision token via IDP client_credentials when not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass

TOKEN = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_WRITE_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)

CATALOG_ID    = os.environ.get("CATALOG_ID", "demo-tiles")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "admin-polygons")

# Tile zoom/x/y for Rome area at z=6
TILE_Z, TILE_X, TILE_Y = 6, 34, 23

headers = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60.0)

print(f"Platform  : {BASE_URL}")
print(f"Catalog   : {CATALOG_ID}")
print(f"Collection: {COLLECTION_ID}")
print(f"Tile      : z={TILE_Z} x={TILE_X} y={TILE_Y}")

## Step 1 — List Tile Matrix Sets

`GET /tiles/tileMatrixSets` returns the available tile matrix sets (e.g.,
WebMercatorQuad, WorldCRS84Quad). This confirms the tiles extension is active.

In [ ]:
r = client.get("/tiles/tileMatrixSets")
print(f"status: {r.status_code}")

_tiles_active = r.status_code == 200

if _tiles_active:
    tms_list = r.json()
    sets = tms_list.get("tileMatrixSets", [])
    print(f"Available Tile Matrix Sets ({len(sets)}):")
    for tms in sets:
        print(f"  id={tms.get('id')}  crs={tms.get('crs', 'n/a')}")
elif r.status_code == 404:
    print("SKIP: /tiles endpoint not found — tiles extension may not be active.")
else:
    print(r.text[:300])

## Step 2 — Create catalog and collection, ingest features

We create a demo catalog and collection via OGC Features, then POST a few
administrative polygons covering central Italy. The tiles extension serves MVT
content directly from these features.

In [ ]:
# --- catalog ---
r = client.post(
    "/features/catalogs",
    content=json.dumps({"id": CATALOG_ID, "title": "Tiles demo catalog"}),
)
print(f"catalog   : {r.status_code}", "(already exists)" if r.status_code == 409 else "")

# --- collection ---
r = client.post(
    f"/features/catalogs/{CATALOG_ID}/collections",
    content=json.dumps({
        "id": COLLECTION_ID,
        "title": "Administrative polygons demo",
    }),
)
print(f"collection: {r.status_code}", "(already exists)" if r.status_code == 409 else "")

# --- items (Italy regions approx polygons) ---
regions = [
    {"id": "lazio",   "name": "Lazio",   "coords": [[11.5,41.2],[13.9,41.2],[13.9,42.9],[11.5,42.9],[11.5,41.2]]},
    {"id": "toscana", "name": "Toscana", "coords": [[9.7,42.4],[12.4,42.4],[12.4,44.5],[9.7,44.5],[9.7,42.4]]},
    {"id": "umbria",  "name": "Umbria",  "coords": [[11.9,42.3],[13.3,42.3],[13.3,43.7],[11.9,43.7],[11.9,42.3]]},
]

for reg in regions:
    feat = {
        "type": "Feature",
        "id": reg["id"],
        "geometry": {"type": "Polygon", "coordinates": [reg["coords"]]},
        "properties": {"name": reg["name"], "country": "IT"},
    }
    r = client.post(
        f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
        content=json.dumps(feat),
    )
    if r.status_code in (200, 201):
        print(f"  item '{reg['id']}': created")
    elif r.status_code == 409:
        print(f"  item '{reg['id']}': already exists")
    else:
        print(f"  item '{reg['id']}': {r.status_code} {r.text[:200]}")

## Step 3 — Fetch an MVT tile

`GET /tiles/catalogs/{dataset}/tiles/{z}/{x}/{y}.mvt` returns a Mapbox Vector
Tile binary. The dataset parameter is the catalog_id. We request z=6, x=34, y=23
which covers central Italy.

The response body is a binary protobuf. We verify the content-type and that the
response is non-empty.

In [ ]:
if not _tiles_active:
    print("SKIP: tiles extension not active — skipping MVT fetch.")
else:
    # Request without Content-Type header (binary response)
    tile_headers = {"Authorization": f"Bearer {TOKEN}"}
    tile_client  = httpx.Client(base_url=BASE_URL, headers=tile_headers, timeout=60.0)

    r = tile_client.get(f"/tiles/catalogs/{CATALOG_ID}/tiles/{TILE_Z}/{TILE_X}/{TILE_Y}.mvt")
    print(f"tile status: {r.status_code}")

    if r.status_code == 200:
        ct = r.headers.get("content-type", "")
        print(f"content-type : {ct}")
        print(f"body size    : {len(r.content)} bytes")
        assert "vector-tile" in ct or "octet-stream" in ct, (
            f"Unexpected content-type: {ct}"
        )
        assert len(r.content) > 0, "Empty tile body"
        print("MVT tile received and content-type verified.")
    elif r.status_code == 204:
        print("Tile returned 204 (empty) — the catalog has no features in this tile.")
        print("Try ingesting more features or a different z/x/y.")
    elif r.status_code == 404:
        print("SKIP: tile endpoint returned 404 — check catalog_id or tile coordinates.")
    else:
        print(r.text[:300])
    tile_client.close()

## Step 4 — Fetch with TileMatrixSet (explicit CRS)

`GET /tiles/catalogs/{dataset}/tiles/{tileMatrixSetId}/{z}/{x}/{y}.mvt` is the
TMS-qualified endpoint. We request the same tile under `WebMercatorQuad`.

In [ ]:
if not _tiles_active:
    print("SKIP: tiles extension not active.")
else:
    tile_headers = {"Authorization": f"Bearer {TOKEN}"}
    tile_client  = httpx.Client(base_url=BASE_URL, headers=tile_headers, timeout=60.0)

    tms_id = "WebMercatorQuad"
    r = tile_client.get(
        f"/tiles/catalogs/{CATALOG_ID}/tiles/{tms_id}/{TILE_Z}/{TILE_X}/{TILE_Y}.mvt"
    )
    print(f"TMS tile status: {r.status_code}")
    if r.status_code == 200:
        print(f"  content-type: {r.headers.get('content-type')}")
        print(f"  size        : {len(r.content)} bytes")
    elif r.status_code == 204:
        print("  204 (empty) — no features in this tile area.")
    elif r.status_code == 404:
        print(f"  404 — TMS '{tms_id}' not registered or tile not found.")
    else:
        print(r.text[:200])
    tile_client.close()

## Teardown

In [ ]:
_TEARDOWN = os.environ.get("TILES_TEARDOWN", "true").lower() == "true"

if _TEARDOWN:
    r = client.delete(f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}")
    print(f"DELETE collection: {r.status_code}")
    r = client.delete(f"/features/catalogs/{CATALOG_ID}")
    print(f"DELETE catalog   : {r.status_code}")
else:
    print("Teardown skipped (set TILES_TEARDOWN=true to enable).")

## Result — Open the map viewer

In [ ]:
client.close()
print(f"Open the map viewer: {BASE_URL}/web/pages/map_viewer?language=en")
print(f"Direct tile URL (z={TILE_Z} x={TILE_X} y={TILE_Y}):")
print(f"  {BASE_URL}/tiles/catalogs/{CATALOG_ID}/tiles/{TILE_Z}/{TILE_X}/{TILE_Y}.mvt")